# PhiStone 分子活性预测

依次运行下方单元格，点击 **选择本地文件 → 开始预测**，自动完成二维图预处理和 PhiSGATv2 活性预测。支持多选 **SDF、CSV、XLSX**；表格需有 `SMILES` 列，`Compound ID` 列可选。

使用项目已有训练/推理环境作为 Notebook 内核，并确保 Jupyter 已启用 `ipywidgets`。默认加载 `PhiSGATv2/training_artifacts/checkpoints/best_checkpoint.pt`；预处理所需的归一化参数和词表等资源沿用项目配置。本 Notebook 不安装依赖或下载权重。

**清除缓存** 会删除本次内核会话产生的全部预处理和预测结果，包括之前批次、失败任务的文件及分子图片。输入文件、模型权重和公共词表保留；需要的结果请先保存。预测值保留模型原始输出，不裁剪到 0～1，也不代表实验测量。

## 1. 加载环境与配置

在项目根目录打开本 Notebook。如果内核工作目录不同，修改 `PROJECT_ROOT` 为服务器上的项目目录。默认每页显示 20 个分子，每行 4 个；配置修改后应先清理缓存并重启内核再重新运行。

In [ ]:
import gc
import math
import shutil
import sys
import uuid
import warnings
from html import escape
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "PhiSGATv2").is_dir():
    raise FileNotFoundError("请在项目根目录打开 Notebook，或修改 PROJECT_ROOT。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("当前内核缺少 ipywidgets，请使用配置好交互控件的 Jupyter 环境。") from exc

from IPython.display import FileLink, Image, display

# 仅屏蔽 PyG GlobalAttention 的弃用提示，保留其他警告。
warnings.filterwarnings(
    "ignore",
    message=r"^'nn\.glob\.GlobalAttention' is deprecated, use 'nn\.aggr\.AttentionalAggregation' instead$",
    category=UserWarning,
    module=r"^torch_geometric\.deprecation$",
)

# 先导入预处理模块，让它按现有逻辑设置数值库线程环境。
from PhiSSeparator import PhiSSeparator_2D_work_data_process as process_2d
from PhiSGATv2 import PhiSGATv2_config as config
from PhiSGATv2 import PhiSGATv2_predict as predictor
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D

CHECKPOINT = Path(config.BEST_CHECKPOINT_PATH)
DEVICE = None  # None 沿用项目配置；也可设置为 "cuda:0" 或 "cpu"。
PAGE_SIZE = 20
GRID_COLUMNS = 4
MOLECULE_SIZE = (300, 240)

print(f"项目目录：{PROJECT_ROOT}")
print(f"预测权重：{CHECKPOINT}")
print(f"推理设备：{DEVICE or config.DEVICE}")

## 2. 定义交互流程

调用顺序为 `process_work_data()` → `predict_activity()`。预处理使用单进程、预测 DataLoader 使用 0 个子进程，保持 Notebook 中的调用简单稳定。分子网格只处理用于绘图的分子副本，不改变预测输入或结果。

每次内核会话创建一个 `notebook_<会话编号>` 子目录，每次预测再创建批次子目录。预处理与预测只读取对应批次。公共片段/原子词表同步仍由底层预处理脚本完成，不属于可删除的缓存。

In [ ]:
class ActivityNotebook:
    def __init__(self):
        self.session_id = f"notebook_{uuid.uuid4().hex}"
        self.roots = {
            "input": Path(config.WORK_INPUT_DIR).resolve(),
            "pyg": Path(config.WORK_PYG_DIR).resolve(),
            "results": Path(config.PREDICTION_OUTPUT_DIR).resolve(),
        }
        self.paths = {key: root / self.session_id for key, root in self.roots.items()}
        self.busy = False
        self.records = []
        self.image_paths = []
        self.upload = widgets.FileUpload(
            accept=".sdf,.csv,.xlsx", multiple=True, description="选择本地文件"
        )
        self.run_button = widgets.Button(description="开始预测", button_style="success", disabled=True)
        self.clear_button = widgets.Button(description="清除缓存", button_style="warning")
        self.page = widgets.Dropdown(options=[], description="结果页：", disabled=True)
        self.status = widgets.HTML(value="请选择文件，然后点击开始预测。")
        self.logs = widgets.Output()
        self.summary = widgets.Output()
        self.grid = widgets.Output()
        locations = "<br>".join(
            f"{label}：<code>{escape(str(self.paths[key]))}</code>"
            for key, label in (("input", "输入"), ("pyg", "预处理"), ("results", "预测结果"))
        )
        self.view = widgets.VBox([
            widgets.HTML(value="<h3>分子活性预测</h3>支持 SDF、CSV、XLSX；可一次选择多个文件。"),
            widgets.HBox([self.upload, self.run_button, self.clear_button]),
            widgets.HTML(value="清除缓存将删除本会话所有批次的 PyG、结果 CSV、图片及处理报告；输入文件保留。"),
            self.status, widgets.HTML(value=locations), self.summary,
            self.page, self.grid,
            widgets.HTML(value="<b>处理日志</b>"), self.logs,
        ])
        self.upload.observe(self._on_upload, names="value")
        self.page.observe(self._show_page, names="value")
        self.run_button.on_click(self.logs.capture()(self._run))
        self.clear_button.on_click(self._clear)

    def _uploads(self):
        value = self.upload.value
        # FileUpload 的两种公开数据格式：文件名字典或文件记录序列。
        if isinstance(value, dict):
            return [(item.get("metadata", {}).get("name", name), item["content"])
                    for name, item in value.items()]
        return [(item["name"], item["content"]) for item in value]

    def _on_upload(self, change):
        self.run_button.disabled = self.busy or not bool(self.upload.value)
        if not self.busy:
            names = [str(name) for name, _ in self._uploads()]
            self.status.value = escape("已选择：" + "、".join(names) if names else "请选择文件。")

    def _set_busy(self, busy):
        self.busy = busy
        self.upload.disabled = busy
        self.run_button.disabled = busy or not bool(self.upload.value)
        self.clear_button.disabled = busy
        self.page.disabled = busy or not bool(self.page.options)

    def _reset_display(self):
        self.records = []
        self.image_paths = []
        self.page.options = []
        self.page.disabled = True
        self.summary.clear_output()
        self.grid.clear_output()

    def _save_grids(self, image_dir):
        image_dir.mkdir(parents=True, exist_ok=True)
        for start in range(0, len(self.records), PAGE_SIZE):
            molecules, legends = [], []
            path = None
            for rank, record in enumerate(self.records[start:start + PAGE_SIZE], start=start + 1):
                try:
                    mol = Chem.MolFromSmiles(record["smiles"])
                    if mol is None:
                        raise ValueError("无法解析 SMILES")
                    for atom in mol.GetAtoms():
                        atom.SetAtomMapNum(0)
                    mol = rdMolDraw2D.PrepareMolForDrawing(mol, addChiralHs=False)
                except Exception as exc:
                    print(f"绘图跳过 {record['compound_id']}：{exc}")
                    continue
                molecules.append(mol)
                # 网格使用排序编号，与下方表格对应，避免长名称或中文字体影响绘图。
                legends.append(f"#{rank} | Activity: {record['predicted_activity']:.4f}")
            if molecules:
                try:
                    columns = min(GRID_COLUMNS, len(molecules))
                    rows = math.ceil(len(molecules) / columns)
                    width, height = MOLECULE_SIZE
                    drawer = rdMolDraw2D.MolDraw2DCairo(columns * width, rows * height, width, height)
                    drawer.drawOptions().prepareMolsBeforeDrawing = False
                    drawer.DrawMolecules(molecules, legends=legends)
                    drawer.FinishDrawing()
                    candidate = image_dir / f"molecules_{start // PAGE_SIZE + 1:03d}.png"
                    candidate.write_bytes(drawer.GetDrawingText())
                    path = candidate
                except Exception as exc:
                    print(f"本页网格绘制失败，预测数值仍保留：{exc}")
            self.image_paths.append(path)

    def _show_page(self, change=None):
        self.grid.clear_output(wait=True)
        index = self.page.value
        if index is None or not self.records:
            return
        start = index * PAGE_SIZE
        records = self.records[start:start + PAGE_SIZE]
        with self.grid:
            path = self.image_paths[index]
            if path is not None and path.is_file():
                display(Image(filename=str(path), width=1000))
            table = pd.DataFrame(records)[["compound_id", "smiles", "predicted_activity"]]
            table.index = range(start + 1, start + len(table) + 1)
            table.index.name = "网格编号"
            display(table)

    def _run(self, button):
        if self.busy or not self.upload.value:
            return
        self._set_busy(True)
        self._reset_display()
        self.logs.clear_output()
        try:
            if not CHECKPOINT.is_file():
                raise FileNotFoundError(f"预测权重不存在：{CHECKPOINT}")
            batch_id = f"batch_{uuid.uuid4().hex[:12]}"
            paths = {key: root / batch_id for key, root in self.paths.items()}
            for path in paths.values():
                path.mkdir(parents=True, exist_ok=False)
            for index, (name, content) in enumerate(self._uploads(), start=1):
                filename = Path(str(name).replace("\\", "/")).name
                if Path(filename).suffix.lower() not in {".sdf", ".csv", ".xlsx"}:
                    raise ValueError(f"不支持的文件类型：{filename}")
                # 同名文件分别放入子目录，避免上传时互相覆盖。
                source_dir = paths["input"] / f"file_{index:03d}"
                source_dir.mkdir()
                (source_dir / filename).write_bytes(bytes(content))

            self.status.value = "正在预处理待测分子……"
            summary = process_2d.process_work_data(
                input_dir=paths["input"], output_dir=paths["pyg"], num_workers=1,
            )
            if not summary["saved_pyg"]:
                raise RuntimeError("没有成功生成可预测的 PyG，请查看处理错误报告。")
            self.status.value = "正在预测分子活性……"
            result_path = paths["results"] / "prediction_results.csv"
            error_path = paths["results"] / "prediction_errors.csv"
            records = predictor.predict_activity(
                data_dir=paths["pyg"], checkpoint_path=CHECKPOINT, device=DEVICE,
                result_path=result_path, error_path=error_path,
                image_dir=None, data_loader_workers=0,
            )
            self.records = sorted(records, key=lambda row: -float(row["predicted_activity"]))
            self.status.value = "正在生成分子网格……"
            self._save_grids(paths["results"] / "molecule_images")
            self.page.options = [(f"第 {i + 1} 页", i) for i in range(len(self.image_paths))]
            self.page.value = 0
            self._show_page()
            error_count = len(pd.read_csv(error_path))
            with self.summary:
                print(f"本批次：{batch_id}")
                print(f"预处理成功 {summary['saved_pyg']} 个；预处理错误 {summary['error_count']} 条；"
                      f"预测成功 {len(records)} 个；预测错误 {error_count} 条。")
                print("网格按预测值降序展示；CSV 保留底层原始顺序及完整数值。")
                print(f"输入：{paths['input']}")
                print(f"预处理：{paths['pyg']}")
                print(f"结果：{paths['results']}")
                for path in (result_path, error_path, paths["pyg"] / "processing_errors.csv"):
                    if path.is_file():
                        # 默认目录位于 Notebook 根目录下，可通过 Jupyter 文件服务打开。
                        if path.is_relative_to(PROJECT_ROOT):
                            display(FileLink(str(path.relative_to(PROJECT_ROOT))))
                        else:
                            print(path)
            self.status.value = f"完成：已预测 {len(records)} 个分子，可选择结果页查看。"
        except (Exception, KeyboardInterrupt) as exc:
            with self.logs:
                print(f"任务未完成：{type(exc).__name__}: {exc}")
                if "paths" in locals():
                    print(f"本批次预处理和结果目录：{paths['pyg']}；{paths['results']}")
            self.status.value = "任务未完成，请查看日志；本次文件可通过清除缓存按钮清理。"
        finally:
            self._set_busy(False)

    def _clear(self, button):
        if self.busy:
            return
        self._set_busy(True)
        failures = []
        try:
            # 仅删除本对象创建的两个会话目录。先核对所有目标，再执行删除。
            targets = [(self.roots[key], self.paths[key]) for key in ("pyg", "results")]
            for root, path in targets:
                if path.is_symlink() or path.resolve().parent != root or path.name != self.session_id:
                    raise ValueError(f"清理目标不是本会话目录：{path}")
            for root, path in targets:
                try:
                    if path.exists():
                        shutil.rmtree(path)
                except OSError as exc:
                    failures.append(f"{path}：{exc}")
            self._reset_display()
            self.logs.clear_output()
            self.upload.value = {} if isinstance(self.upload.value, dict) else ()
            gc.collect()
            if failures:
                with self.logs:
                    print("部分文件未能清除，可再次点击清除缓存：\n" + "\n".join(failures))
                self.status.value = "部分缓存未清除，请查看日志。"
            else:
                self.status.value = "本会话预处理与预测缓存已清除，输入文件已保留。可重新选择文件。"
        except Exception as exc:
            self.status.value = escape(f"清理停止：{exc}")
        finally:
            self._set_busy(False)

## 3. 打开操作面板

运行此单元格后，选择文件并点击 **开始预测**。预测期间按钮暂时禁用；计算日志显示在面板下方。可通过结果页下拉框查看全部分子，结果 CSV 包含完整预测值。

重复运行本单元格会继续使用当前会话。重启内核后会创建新会话，旧会话磁盘文件不会自动清除；建议在重启前保存所需结果并点击 **清除缓存**。该按钮不会删除 Notebook 文件本身，也不会清除已经保存到 Notebook 文件中的历史输出；如需去除已保存输出，请使用 Jupyter 的清除输出功能后保存。

In [ ]:
# 复用当前面板，避免重复运行显示单元格时丢失缓存清理范围。
if "phistone_activity_ui" not in globals():
    phistone_activity_ui = ActivityNotebook()
display(phistone_activity_ui.view)

## 文件位置与依赖说明

| 内容 | 默认目录（其下创建会话和批次子目录） |
| --- | --- |
| 上传输入 | `Datas/work_datas/input_unknownactivity/` |
| 预处理 PyG 和处理报告 | `Datas/work_datas/processed_2d/` |
| 预测 CSV、错误报告、网格 PNG | `Datas/work_datas/PhiSGATv2_prediction_results/` |

目录读取自 `PhiSGATv2_config.py`。默认模型权重为 `PhiSGATv2/training_artifacts/checkpoints/best_checkpoint.pt`。XLSX 读取沿用项目中的 `openpyxl` 依赖；Notebook 另外需要 `IPython`、`ipywidgets` 和支持控件的 Jupyter 前端。若只显示控件名称而没有按钮，请检查当前内核和前端的控件支持。

Notebook 在服务器运行时，文件按钮选择的是浏览器所在电脑的本地文件，内容会传到服务器保存。本次预测使用新批次目录，不会扫描默认根目录中的旧输入或旧 PyG。清除缓存不会撤销预处理对公共词表的同步，也不会删除上传后保留的输入文件。

接口参考：[ipywidgets 文件上传](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html#File-Upload)、[RDKit 分子绘图](https://rdkit.org/docs/source/rdkit.Chem.Draw.rdMolDraw2D.html)。